# 🚀 Phase 1: Probing & Steering Vector Estimation
**Paper Title:** Early-Stopping Activation Steering for Hallucination Mitigation in Vietnamese Domain-Specific RAG
**Dataset:** 14,700 Specialized Medical QA (`vietnamese_medical_halueval_15k_specialized.json`)
**Splits:** 10,290 Train / 2,205 Validation / 2,205 Test (Question-Disjoint)
**Model:** `Qwen/Qwen2.5-7B-Instruct` (4-bit NF4 Quantization)

---

In [1]:
# Cell 1: Environment Setup & Dependencies
!pip install -q bitsandbytes accelerate transformers torch scikit-learn rouge-score tqdm
print('✅ Dependencies successfully installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 36.6 MB/s eta 0:00:00
✅ Dependencies successfully installed!


In [2]:
# Cell 2: Data Loading & Path Quetting
import os, json, glob, random
import numpy as np
import torch

DATA_FILENAME = 'vietnamese_medical_halueval_15k_specialized.json'
search_paths = [
    f'/kaggle/input/**/{DATA_FILENAME}',
    f'/kaggle/input/{DATA_FILENAME}',
    f'data/{DATA_FILENAME}',
    f'./{DATA_FILENAME}',
    f'E:/Paper_Steering_VN_15K/data/{DATA_FILENAME}'
]

data_path = None
for pattern in search_paths:
    matches = glob.glob(pattern, recursive=True)
    if matches:
        data_path = matches[0]
        break

if not data_path:
    raise FileNotFoundError(f'❌ Could not locate {DATA_FILENAME}. Please upload the dataset to Kaggle Input.')

print(f'✅ Located dataset at: {data_path}')
with open(data_path, 'r', encoding='utf-8') as f:
    raw_dataset = json.load(f)

print(f'📊 Total QA Records Loaded: {len(raw_dataset):,}')

✅ Located dataset at: /kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json
📊 Total QA Records Loaded: 14,700


In [3]:
# Cell 3: Question-Disjoint Partitioning (70% Train / 15% Val / 15% Test)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

shuffled_records = list(raw_dataset)
random.shuffle(shuffled_records)

n_total = len(shuffled_records)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
n_test = n_total - n_train - n_val

train_records = shuffled_records[:n_train]
val_records = shuffled_records[n_train:n_train + n_val]
test_records = shuffled_records[n_train + n_val:]

print(f'📌 Train Split      : {len(train_records):,} records (70%)')
print(f'📌 Validation Split : {len(val_records):,} records (15%)')
print(f'📌 Test Split       : {len(test_records):,} records (15%)')

📌 Train Split      : 10,290 records (70%)
📌 Validation Split : 2,205 records (15%)
📌 Test Split       : 2,205 records (15%)


In [4]:
# Cell 4: Initialize Qwen2.5-7B-Instruct Engine in 4-bit Quantization
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

print(f'⌛ Loading model {MODEL_NAME} in 4-bit NF4...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model.eval()
print('✅ Qwen2.5-7B-Instruct successfully loaded on GPU!')

⌛ Loading model Qwen/Qwen2.5-7B-Instruct in 4-bit NF4...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Qwen2.5-7B-Instruct successfully loaded on GPU!


In [5]:
# Cell 5: Layer-wise Activation Probing (Optimized Sample Count)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
from tqdm import tqdm
import numpy as np
import torch

PROMPT_TEMPLATE = """Dựa vào ngữ cảnh y học sau đây, hãy trả lời câu hỏi:
Ngữ cảnh: {context}
Câu hỏi: {question}
Trả lời: """

print('⌛ Extracting training activations across layers for probing...')
probe_sample_count = min(400, len(train_records))
probe_records = train_records[:probe_sample_count]

target_layers = [4, 8, 12, 16, 20, 24, 27]
activations_pos = {l: [] for l in target_layers}
activations_neg = {l: [] for l in target_layers}

with torch.no_grad():
    for rec in tqdm(probe_records, desc='Extracting Probing Activations'):
        ctx = rec.get('knowledge_context', rec.get('context', ''))
        q = rec['question']
        a_pos = rec['right_answer']
        a_neg = rec['hallucinated_answer']
        
        text_pos = PROMPT_TEMPLATE.format(context=ctx, question=q) + a_pos
        text_neg = PROMPT_TEMPLATE.format(context=ctx, question=q) + a_neg
        
        inputs_pos = tokenizer(text_pos, return_tensors='pt', padding=False).to('cuda')
        inputs_neg = tokenizer(text_neg, return_tensors='pt', padding=False).to('cuda')
        
        out_pos = model(**inputs_pos, output_hidden_states=True)
        out_neg = model(**inputs_neg, output_hidden_states=True)
        
        for l in target_layers:
            act_p = out_pos.hidden_states[l][0, -1, :].detach().float().cpu().numpy()
            act_n = out_neg.hidden_states[l][0, -1, :].detach().float().cpu().numpy()
            activations_pos[l].append(act_p)
            activations_neg[l].append(act_n)

print('\n📊 --- LAYER-WISE PROBING RESULTS ---')
best_layer = 16
best_f1 = 0.0

for l in target_layers:
    X = np.vstack([activations_pos[l], activations_neg[l]])
    y = np.array([1]*len(activations_pos[l]) + [0]*len(activations_neg[l]))
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X, y)
    preds_prob = clf.predict_proba(X)[:, 1]
    preds = clf.predict(X)
    auc = roc_auc_score(y, preds_prob)
    f1 = f1_score(y, preds)
    print(f'Layer {l:2d} | AUROC: {auc:.4f} | F1-Score: {f1:.4f}')
    if f1 > best_f1:
        best_f1 = f1
        best_layer = l

print(f'\n🏆 Selected Optimal Intervention Layer: Module Index {best_layer} (F1: {best_f1:.4f})')

⌛ Extracting training activations across layers for probing...


Extracting Probing Activations: 100%|██████████| 400/400 [38:49<00:00,  5.82s/it]



📊 --- LAYER-WISE PROBING RESULTS ---
Layer  4 | AUROC: 0.9944 | F1-Score: 0.9558
Layer  8 | AUROC: 1.0000 | F1-Score: 1.0000
Layer 12 | AUROC: 1.0000 | F1-Score: 1.0000
Layer 16 | AUROC: 1.0000 | F1-Score: 1.0000
Layer 20 | AUROC: 1.0000 | F1-Score: 1.0000
Layer 24 | AUROC: 1.0000 | F1-Score: 1.0000
Layer 27 | AUROC: 1.0000 | F1-Score: 1.0000

🏆 Selected Optimal Intervention Layer: Module Index 8 (F1: 1.0000)


In [6]:
# Cell 6: Estimate Contrastive Steering Vector (v_steer)
print(f'⌛ Estimating raw contrastive direction at Layer {best_layer} on Train Set...')

mean_pos = np.mean(activations_pos[best_layer], axis=0)
mean_neg = np.mean(activations_neg[best_layer], axis=0)
v_raw = mean_pos - mean_neg
v_norm = np.linalg.norm(v_raw)
v_steer_np = v_raw / v_norm

v_steer = torch.tensor(v_steer_np, dtype=torch.bfloat16, device='cuda')
v_rand = torch.randn_like(v_steer)
v_rand = (v_rand / torch.norm(v_rand, p=2)).to(torch.bfloat16)

print(f'✅ Steering Vector v_steer computed & unit-normalized (dim={v_steer.shape[0]})')

⌛ Estimating raw contrastive direction at Layer 8 on Train Set...
✅ Steering Vector v_steer computed & unit-normalized (dim=3584)


In [7]:
# Cell 7: Generation Hook Implementation & Factorial Grid Sweep on Validation Set
import time
import torch
import numpy as np
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

class EarlyStoppingSteeringHook:
    def __init__(self, layer_idx, v_vector, alpha=20.0, K=8):
        self.layer_idx = layer_idx
        self.v_vector = v_vector
        self.alpha = alpha
        self.K = K
        self.step_counter = 0
        self.handle = None
        
    def hook_fn(self, module, inputs, output):
        if self.step_counter < self.K:
            if isinstance(output, tuple):
                h = output[0]
                v = self.v_vector.to(h.device)
                h[:, -1, :] = h[:, -1, :] + self.alpha * v
                output = (h,) + output[1:]
            else:
                v = self.v_vector.to(output.device)
                output[:, -1, :] = output[:, -1, :] + self.alpha * v
        self.step_counter += 1
        return output
        
    def register(self, model):
        layer_module = model.model.layers[self.layer_idx]
        self.step_counter = 0
        self.handle = layer_module.register_forward_hook(self.hook_fn)
        
    def remove(self):
        if self.handle:
            self.handle.remove()
            self.handle = None

print('⌛ Running Factorial Grid Sweep on Validation Set...')
val_sample_subset = val_records[:50]

sweep_alphas = [5.0, 10.0, 15.0, 20.0]
sweep_Ks = [1, 4, 8, 16, 999]
best_combo = (20.0, 8)
best_val_rouge = 0.0

grid_results = {}
for a in sweep_alphas:
    for k in sweep_Ks:
        k_label = 'Full' if k == 999 else k
        rouges = []
        for rec in val_sample_subset:
            ctx = rec.get('knowledge_context', rec.get('context', ''))
            q = rec['question']
            ref = rec['right_answer']
            prompt = PROMPT_TEMPLATE.format(context=ctx, question=q)
            inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
            
            hook = EarlyStoppingSteeringHook(best_layer, v_steer, alpha=a, K=k)
            hook.register(model)
            with torch.no_grad():
                torch.manual_seed(42)
                out_ids = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.1, top_p=0.85)
            hook.remove()
            
            gen_text = tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            r_score = scorer.score(ref, gen_text)['rougeL'].fmeasure * 100
            rouges.append(r_score)
        
        mean_r = np.mean(rouges)
        grid_results[(a, k_label)] = mean_r
        print(f'Grid (alpha={a:4.1f}, K={str(k_label):>4}) | ROUGE-L: {mean_r:.2f}%')
        if mean_r > best_val_rouge:
            best_val_rouge = mean_r
            best_combo = (a, k)

print(f'\n🏆 Selected Peak Grid Config: alpha={best_combo[0]}, K={best_combo[1]}')

⌛ Running Factorial Grid Sweep on Validation Set...
Grid (alpha= 5.0, K=   1) | ROUGE-L: 31.51%
Grid (alpha= 5.0, K=   4) | ROUGE-L: 31.33%
Grid (alpha= 5.0, K=   8) | ROUGE-L: 32.04%
Grid (alpha= 5.0, K=  16) | ROUGE-L: 33.01%
Grid (alpha= 5.0, K=Full) | ROUGE-L: 32.24%
Grid (alpha=10.0, K=   1) | ROUGE-L: 32.33%
Grid (alpha=10.0, K=   4) | ROUGE-L: 32.68%
Grid (alpha=10.0, K=   8) | ROUGE-L: 32.99%
Grid (alpha=10.0, K=  16) | ROUGE-L: 33.06%
Grid (alpha=10.0, K=Full) | ROUGE-L: 33.72%
Grid (alpha=15.0, K=   1) | ROUGE-L: 32.64%
Grid (alpha=15.0, K=   4) | ROUGE-L: 32.25%
Grid (alpha=15.0, K=   8) | ROUGE-L: 32.97%
Grid (alpha=15.0, K=  16) | ROUGE-L: 33.77%
Grid (alpha=15.0, K=Full) | ROUGE-L: 33.65%
Grid (alpha=20.0, K=   1) | ROUGE-L: 32.87%
Grid (alpha=20.0, K=   4) | ROUGE-L: 33.10%
Grid (alpha=20.0, K=   8) | ROUGE-L: 33.90%
Grid (alpha=20.0, K=  16) | ROUGE-L: 33.33%
Grid (alpha=20.0, K=Full) | ROUGE-L: 34.88%

🏆 Selected Peak Grid Config: alpha=20.0, K=999


In [8]:
# Cell 8: Save Checkpoint Artifacts for Phase 2 Evaluation
import json, torch

print('💾 Exporting Steering Vectors and Optimal Metadata Config...')

# 1. Save Steering Vectors to CPU tensors
torch.save(v_steer.cpu(), 'v_steer.pt')
torch.save(v_rand.cpu(), 'v_rand.pt')

# 2. Save metadata JSON
steering_config = {
    'best_layer': int(best_layer),
    'best_alpha': float(best_combo[0]),
    'best_K': int(best_combo[1]) if best_combo[1] != 999 else 999
}

with open('steering_config.json', 'w', encoding='utf-8') as f:
    json.dump(steering_config, f, indent=2)

print('✅ Successfully exported Phase 1 artifacts:')
print('   - v_steer.pt')
print('   - v_rand.pt')
print('   - steering_config.json')
print(f'   - Config values: Layer {best_layer}, Alpha {best_combo[0]}, K {best_combo[1]}')

💾 Exporting Steering Vectors and Optimal Metadata Config...
✅ Successfully exported Phase 1 artifacts:
   - v_steer.pt
   - v_rand.pt
   - steering_config.json
   - Config values: Layer 8, Alpha 20.0, K 999
